# MiniMax H3：G4 96GB 本地推理
先完成 `01_prepare_cpu.ipynb`。选择 **G4 GPU**；查看系统 RAM，默认 staged 模式预检要求至少 **85 GiB 可用 RAM**。
这里会重新拉取代码并用 Drive 轮子离线恢复环境，随后做 GPU 检查和短片推理。
GPU 分配期间安装、读取权重、等待等也可能消耗计算单元；本方案减少准备和调试耗时，不能让启动开销归零。
默认不调用 MiniMax 付费 API；2K 服务不在此本地流程内。

In [ ]:
# 已填写当前仓库地址；如 fork 到其他账号，请修改此项。
REPO_URL = "https://github.com/simonFelix-Ai/colab.git"
REPO_REF = "main"  # 建议确认后改成自己的提交 SHA
PERSIST_ROOT = "/content/drive/MyDrive/minimax-h3-colab"
WORKFLOW = "t2va"  # t2va / fl2va 共用权重；ref2va 需要另一个 transformer

In [ ]:
import os, subprocess, sys
from pathlib import Path
assert REPO_URL.startswith("https://github.com/"), "请填写有效的 GitHub 仓库 URL"
assert sys.version_info[:2] == (3, 12), "本项目使用 Python 3.12，请切到兼容的 Colab 运行时"
PROJECT = Path("/content/minimax-h3-colab")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(PROJECT)], check=True)
else:
    assert subprocess.check_output(["git", "-C", str(PROJECT), "remote", "get-url", "origin"], text=True).strip() == REPO_URL, "现有目录属于其他仓库"
subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(PROJECT)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("代码版本:", COMMIT)
def run(*args):
    subprocess.run([str(x) for x in args], check=True, cwd=PROJECT)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PERSIST = Path(PERSIST_ROOT)
PERSIST.mkdir(parents=True, exist_ok=True)
import hashlib
ENV_KEY = hashlib.sha256((PROJECT / "requirements-colab.txt").read_bytes()).hexdigest()[:12]
ENV_CACHE = PERSIST / ("environment-py312-" + ENV_KEY)
MODEL_DIR = PERSIST / "models/MiniMax-H3"
PY = PROJECT / ".venv/bin/python"

## 1. 离线恢复环境
没有缓存时立即停止，返回 CPU 笔记本准备。独立虚拟环境通过子进程使用，无须重启 notebook 内核。

In [ ]:
run(sys.executable, "scripts/environment.py", "install", "--cache", ENV_CACHE)
run(PY, "scripts/validate_runtime.py")

## 2. 设置生成配置
首次用 smoke。staged 文生视频先编码提示词，再释放编码器，然后加载视频模型，减少 RAM 峰值。
image_to_video 使用 auto_offload，预检要求至少 150 GiB 可用 RAM；先准备好图片路径。
默认直接读取 Drive 权重。可选复制到本地 SSD，需额外约 140 GiB 空闲磁盘，复制本身也占 GPU 会话时间。

In [ ]:
CONFIG_PATH = PROJECT / "configs/smoke.json"  # 成功后改 configs/quality.json
OUTPUT_DIR = PERSIST / "outputs"
USE_LOCAL_SSD = False
if USE_LOCAL_SSD:
    import shutil
    local_model = Path("/content/h3-model")
    need = sum(p.stat().st_size for p in MODEL_DIR.rglob("*") if p.is_file())
    assert shutil.disk_usage("/content").free > need + 10 * 1024**3, "本地磁盘空间不足"
    shutil.copytree(MODEL_DIR, local_model, dirs_exist_ok=True)
    MODEL_DIR = local_model
run(PY, "scripts/h3.py", "validate-config", "--config", CONFIG_PATH)
run(PY, "scripts/h3.py", "check", "--model-dir", MODEL_DIR, "--config", CONFIG_PATH)

## 3. 生成视频和原生音频
124 帧约 5.17 秒。smoke 的 9 个 sigma 点执行 8 次模型预测，主要验证流程；正式配置用更多步数。输出直接保存到 Drive，附 JSON 参数、耗时、显存与错误记录。每次点击都会新建一条生成任务。

In [ ]:
run(PY, "scripts/h3.py", "run", "--model-dir", MODEL_DIR, "--output-dir", OUTPUT_DIR, "--config", CONFIG_PATH)

## 4. 预览与结束
确认输出保存在 Drive 后执行最后一格释放运行时；不会自动重试或在后台循环生成。

In [ ]:
from IPython.display import Video, display
videos = sorted(OUTPUT_DIR.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
assert videos, "还没有生成 MP4"
display(Video(str(videos[-1]), embed=True))
print(videos[-1])

In [ ]:
# 完成后手动执行这一格，结束当前运行时。
from google.colab import runtime
runtime.unassign()